# Tutorial 3: Model Diagnosis

After training a deepSCENIC model, the critical question is: **did the model learn biologically meaningful regulatory patterns?**

This tutorial covers how to extract and visualize the model's learned representations to assess whether it has captured cell-type-specific gene regulation. We focus on two key embedding spaces:

1. **Enhancer activity (enh_act)**: Tests whether the E1 matrix learned meaningful TF→region binding patterns
2. **Predicted gene regulatory signal (z_rna)**: Tests the full GRN pipeline (E1 + E2)

If cells cluster by type in these spaces, the model has learned to distinguish cell types based on their regulatory programs.

## Setup

First, let's import the required packages. We'll use scanpy for UMAP visualization since deepSCENIC stores embeddings in the standard `mdata.obsm` format that scanpy expects.

In [1]:
import torch
import scanpy as sc

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

deepSCENIC version: 0.1.0
Using device: cuda


## Load Model and Data

We load the preprocessed MuData from {doc}`01_data_preparation` and the trained model from {doc}`02_training`. Make sure your MuData includes cell annotations (like cell type or cell state) that you can use to color the UMAP plots.

In [2]:
import os

data_dir = "../../../../data/deepSCENIC/MM_lines/"
assert os.path.exists(data_dir), "Data directory not found"

# Load preprocessed MuData
mdata = ds.read(data_dir + "preprocessed_data.h5mu")

# Load trained model
model = ds.tl.load_model(data_dir + "weights/converted_model.pt", device=device)

# Verify cell annotations are available (if you have them)
print(f"\nCell annotations: {list(mdata.obs.columns)}")
if "cell_state" in mdata.obs:
    print(f"Cell states: {mdata.obs['cell_state'].value_counts().to_dict()}")

deepscenic.io - INFO - Loaded ../../../../data/deepSCENIC/MM_lines/preprocessed_data.h5mu: 936 cells, 12156 genes, 281820 regions
deepscenic.tl - INFO - Loaded model from ../../../../data/deepSCENIC/MM_lines/weights/converted_model.pt: 11717 genes, 1386 TFs, 273545 regions



Cell annotations: ['cell_line', 'cell_state', 'split']
Cell states: {'INT': 358, 'MEL': 318, 'MES': 260}


```{tip}
If you encounter GPU memory errors during inference, reduce the `batch_size` parameter in `ds.tl.to_latent()`. The default (256) works for most cases, but very large datasets may require smaller batches.
```

## Extract Model Embeddings

The deepSCENIC model produces several intermediate representations as it transforms TF expression into gene expression predictions:

| Embedding | Shape | What it tests |
|-----------|-------|---------------|
| **z_tf** | (n_cells, n_tfs) | Encoder only - latent TF activity |
| **enh_act** | (n_cells, n_regions) | Encoder + E1 - enhancer activity from TF binding |
| **z_rna** | (n_cells, n_genes) | Full GRN (E1 + E2) - gene regulatory signal |
| **rna_rec** | (n_cells, n_genes) | Full model - reconstructed gene expression |
| **atac_rec** | (n_cells, n_regions) | Full model - reconstructed chromatin accessibility |

### Which Embeddings Matter for Diagnosis?

The key diagnostic embeddings are **enh_act** and **z_rna** because they test whether the learned GRN matrices (E1 and E2) capture biology:

- **enh_act = z_tf @ E1.T**: If enhancer activity clusters by cell type, the E1 matrix learned cell-type-specific TF→region binding
- **z_rna = enh_act @ E2**: If predicted gene signal clusters by cell type, the full GRN pipeline works

z_tf alone only tests the encoder, not the GRN. A model could have good z_tf clustering but poor E1/E2 matrices.

The {func}`~deepscenic.tl.to_latent` function extracts all embeddings and stores them in `mdata.obsm` with the prefix `X_deepscenic_`.

In [ ]:
# Extract all embeddings in one pass and store in mdata.obsm
# Use batch_size to avoid GPU memory issues
ds.tl.to_latent(model, mdata, batch_size=64)

# Embeddings are now stored in mdata.obsm with X_deepscenic_ prefix
print("Extracted embeddings (stored in mdata.obsm):")
for key in mdata.obsm.keys():
    if key.startswith("X_deepscenic_"):
        print(f"  {key}: {mdata.obsm[key].shape}")

## Visualizing Enhancer Activity

The **enhancer activity (enh_act)** embedding is the primary diagnostic. It shows whether the model learned cell-type-specific enhancer regulation through TF binding (the E1 matrix).

### Why enh_act is the Key Diagnostic

1. **It tests the E1 matrix**: enh_act = z_tf @ E1.T, so good clustering means E1 learned meaningful TF→region binding patterns
2. **It's biologically interpretable**: Each dimension is a genomic region, and cell-type-specific enhancer activity is a known biological phenomenon
3. **It's what makes deepSCENIC unique**: The enhancer-mediated regulation is the core of the model

### What to Expect

For a well-trained model on the MM_lines dataset, you should see:
- **MEL (melanocytic)** and **MES (mesenchymal)** cells forming distinct clusters
- **INT (intermediate)** cells positioned between them
- Clear separation by cell state, driven by different enhancer activity profiles

Since enh_act is high-dimensional (~280k regions), we use PCA before computing the UMAP:

In [ ]:
# Create AnnData from enhancer activity for visualization
import anndata as ad

adata_enh = ad.AnnData(mdata.obsm["X_deepscenic_enh_act"])
adata_enh.obs = mdata.obs.copy()

# PCA first (enh_act is high-dimensional)
sc.pp.pca(adata_enh, n_comps=50)
sc.pp.neighbors(adata_enh)
sc.tl.umap(adata_enh)

# Visualize colored by cell state
sc.pl.umap(adata_enh, color="cell_state", title="Enhancer Activity (enh_act)")

## Visualizing Predicted Gene Signal

The **predicted gene regulatory signal (z_rna)** tests the full GRN pipeline. It's computed as enh_act @ E2, so good clustering here means both E1 and E2 matrices are working together to predict cell-type-specific gene regulation.

This is the most comprehensive test of model quality - it shows whether the learned TF→region→gene cascade produces biologically meaningful predictions.

In [ ]:
# Create AnnData from predicted gene signal
adata_rna = ad.AnnData(mdata.obsm["X_deepscenic_z_rna"])
adata_rna.obs = mdata.obs.copy()

# PCA first (z_rna is high-dimensional ~12k genes)
sc.pp.pca(adata_rna, n_comps=50)
sc.pp.neighbors(adata_rna)
sc.tl.umap(adata_rna)

# Visualize colored by cell state
sc.pl.umap(adata_rna, color="cell_state", title="Predicted Gene Signal (z_rna)")

### Comparing Both Embeddings

Let's visualize both key embeddings side-by-side:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Enhancer activity UMAP
sc.pl.umap(adata_enh, color="cell_state", ax=axes[0], show=False, 
           title="Enhancer Activity (enh_act)")

# Predicted gene signal UMAP  
sc.pl.umap(adata_rna, color="cell_state", ax=axes[1], show=False,
           title="Predicted Gene Signal (z_rna)")

plt.tight_layout()
plt.show()

Both embeddings should show consistent cell-type structure:

- **If both cluster well**: The model learned a coherent TF→region→gene regulatory cascade
- **If enh_act clusters but z_rna doesn't**: The E1 matrix is good, but E2 (region→gene) needs improvement
- **If neither clusters well**: The model may need more training or there's a data preprocessing issue

The fact that both spaces show similar structure is a strong indication that the model learned biologically meaningful regulation.

## Optional: TF Activity Space (z_tf)

The z_tf embedding represents the encoder's latent TF activity before multiplication by E1. While it's not the primary diagnostic, it can be useful for:

- **Debugging**: If z_tf clusters well but enh_act doesn't, the problem is in E1
- **TF interpretation**: z_tf dimensions directly correspond to TFs, making it easier to identify which TFs drive clustering

Since z_tf is lower-dimensional (~1,400 TFs), you can use it directly without PCA:

```python
# UMAP directly on z_tf (no PCA needed)
sc.pp.neighbors(mdata, use_rep="X_deepscenic_z_tf")
sc.tl.umap(mdata)
sc.pl.umap(mdata, color="cell_state", title="TF Activity (z_tf)")
```

However, remember that **good z_tf clustering doesn't guarantee a good model** - you need to check enh_act and z_rna to verify the GRN matrices are working.

## Interpreting Your Results

### Good Signs ✓

A well-trained model will show:

- **Distinct clusters in both enh_act and z_rna**: Cells of the same type group together
- **Consistent structure across both spaces**: Similar cluster arrangements indicate a coherent GRN
- **Biologically meaningful organization**: Intermediate states between extremes (like INT between MEL and MES)
- **Biology-driven clustering, not batch effects**: Clusters reflect cell types, not samples

### Warning Signs ⚠

| Pattern | Likely Cause | Solution |
|---------|--------------|----------|
| **Random mixing in enh_act** | E1 matrix not learning TF→region patterns | More training, check TF marking |
| **Good enh_act but poor z_rna** | E2 matrix (region→gene) not working | Check R2G penalty matrix, try E2 finetuning |
| **Batch/sample clustering instead of cell types** | Batch effects dominating | Consider batch correction |
| **Single blob with no structure** | Training collapsed | Check learning rate, increase epochs |

### Troubleshooting Tips

1. **Check embeddings exist**: `print(list(mdata.obsm.keys()))` should show `X_deepscenic_*` keys
2. **Verify TF marking**: `mdata['rna'].var['is_tf'].sum()` should be ~1,000-2,000 TFs
3. **Try more training**: Early stopping may have triggered too soon
4. **Check training curves**: Use `ds.pl.loss_curves(model)` if available

## Training Curves (For Newly Trained Models)

If you trained the model yourself using `ds.tl.train()` (rather than loading a converted legacy model), you can examine the training history to assess convergence.

```python
# Plot training curves (only available for models trained with ds.tl.train())
ds.pl.loss_curves(model)
```

### What to Look For

- **Decreasing loss**: Both train and test losses should decrease over epochs
- **No large gap between train/test**: A large gap suggests overfitting
- **Stable convergence**: Loss should stabilize, not oscillate wildly
- **All loss components behaving**: Check RNA loss, ATAC loss, KL divergence, and sparsity terms

```{note}
Legacy models loaded with `ds.tl.load_legacy_model()` don't have training history, so this function won't work for them.
```

## TF Activity Profiles by Cell Type

Once you've confirmed good clustering in enh_act and z_rna, you can dive deeper into which TFs drive the cell-type differences. A clustered heatmap of TF activity scores reveals the regulatory programs that distinguish each cell type.

### Computing TF Activity Scores

The TF activity score for each cell type is computed by:
1. Identifying active enhancers per cell type (using Otsu thresholding on enhancer activity)
2. Weighting TF activity by their binding to these active enhancers (via the E1 matrix)

This gives a biologically meaningful measure of how important each TF is for each cell type's regulatory program.

In [ ]:
# Compute TF activity scores for visualization
enh_activity = ds.tl.compute_celltype_enhancer_activity(model, mdata, "cell_state")
active_enh = ds.tl.identify_active_enhancers(enh_activity)
tf_scores = ds.tl.compute_tf_activity_scores(model, mdata, "cell_state", active_enh)

# Show top 50 most variable TFs across cell states
ds.pl.tf_activity_clustermap(tf_scores, xlabel="cell_state", top_n_tfs=50)

In the heatmap above, each row is a TF and each column is a cell type. TFs are clustered by their activity patterns, so TFs with similar cell-type-specificity appear together. Look for:

- **Cell-type-specific TFs**: TFs that are highly active in one cell type but not others
- **Shared TFs**: TFs active across multiple cell types (these may be housekeeping regulators)
- **Known biology**: For melanoma, you might expect MITF-family TFs to be active in MEL cells, and AP-1/inflammatory TFs in MES cells

This analysis is explored in more detail in {doc}`04_grn_analysis`.

## Summary

In this tutorial, we've covered:

1. **Extracting embeddings** from the trained model using `ds.tl.to_latent()`
2. **Visualizing enhancer activity (enh_act)** - the primary diagnostic that tests the E1 matrix
3. **Visualizing predicted gene signal (z_rna)** - tests the full GRN pipeline (E1 + E2)
4. **Comparing both embeddings** to verify consistent cell-type structure
5. **Interpreting results** with guidance on good signs, warning signs, and troubleshooting

The key insight: **check enh_act and z_rna, not just z_tf**. Good clustering in these downstream embeddings confirms that the learned GRN matrices (E1, E2) capture biologically meaningful regulation.

## Next Steps

Now that you've validated your model:

- **{doc}`04_grn_analysis`**: Extract and analyze the learned gene regulatory network
- **{doc}`05_perturbation_analysis`**: Simulate in-silico TF knockouts and overexpression